In [1]:
%%capture
import os

!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [7]:
# get the huggingface access token
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
huggingface_token= user_secrets.get_secret("huggingface")

**LoRA adapter callback** \
This function uploads the LoRA adapter files from each saved checkpoint duirng trainint to the root of a Hugginface model reposititory. This keeps the repo updated with the latest LoRA weights without requiring manual uploads.

In [10]:
import os
from transformers import TrainerCallback
from huggingface_hub import upload_folder

class AutoUploadCallback(TrainerCallback):
    def __init__(self, repo_id):
        # HF rep to upload to (e.g. "username/model-name")
        self.repo_id = repo_id
        self.token = huggingface_token   # Must be set in Kaggle Secrets

    def on_save(self, args, state, control, **kwargs):
        """
        This keeps the repo updated with the latest LoRA weights
        without requiring manual uploads.
        """

        # Build the full path to the current checkpoint dic (e.g. output/checkpoint-500)
        checkpoint_path = os.path.join(
            args.output_dir,
            f"checkpoint-{state.global_step}"
        )

        # Check if the the checkpoint dic exists 
        if os.path.isdir(checkpoint_path):
            print(f"\n🤖 Uploading {checkpoint_path} to HuggingFace repo root...")

            # Upload ONLY LoRA-specific files to the repository root
            upload_folder(
                repo_id=self.repo_id, # Target HF rep
                folder_path=checkpoint_path, # Folder to upload from
                path_in_repo=".",               # <-- upload directly to repo root
                repo_type="model",
                token=self.token,               # <-- required for auth
                commit_message=f"Overwrite LoRA files ({state.global_step})",
                allow_patterns=[
                    "adapter_model.safetensors", # LoRA weights 
                    "adapter_config.json", # LoRA config 
                ],
            )

            print("✅ Successfully uploaded to repo root!\n")

        return control

In [13]:
# Download HuggingFace model repo
from huggingface_hub import snapshot_download
base_path = snapshot_download("fredrikschultz/flytech_lora")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

In [14]:
import os
from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel

# Load a model and tokenizer from the folder path stored in base_path.
model, tokenizer = FastLanguageModel.from_pretrained(
    base_path,          # Local directory containing model files
    max_seq_length=2048,# Set maximum sequence length for the model
    load_in_4bit=True,  # Load the model in 4-bit mode to save GPU memory
)

# Print which layers/parameters in the model are trainable.
model.print_trainable_parameters()


==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth 2025.11.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


# Data Handling
In the first cell above we create a function that will format the dataset for chat based fine tuning, it prepares the dataset training a chat-style language model (e.g. Llama 3.2). The following cell import the dataset that is going to be used in this finetuning for example "flytech/python-codes-25k" which is create for training a code expert LLM. 

In [16]:
from unsloth.chat_templates import get_chat_template

# Apply the correct chat template for the model family
tokenizer = get_chat_template(tokenizer, "llama-3.2")

# This function converts your dataset rows into properly formatted chat-style training text for supervised fine-tuning.
def formatting_prompts_func(examples):
    # Extract columns from the dataset
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]

    conversations = []
    
    # Loop through each training example
    for instruction, input_text, output in zip(instructions, inputs, outputs):

        # Build the user message:
        if input_text and input_text.strip() != "":
            user_msg = instruction + "\n\n" + input_text
        else:
            user_msg = instruction

        # Create a conversation in chat format:
        # - user message
        # - assistant response (the label)
        convo = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": output},
        ]

        # Convert the conversation into a training string using the model's chat template.
        text = tokenizer.apply_chat_template(
            convo,
            tokenize=False,            # return plain text, not token IDs
            add_generation_prompt=False # do not append assistant generation tag
        )

        conversations.append(text)

    # Return a new column with the formatted training text
    return {"text": conversations}


In [17]:
from datasets import load_dataset

# Load the training split of your Hugging Face dataset
dataset = load_dataset("flytech/python-codes-25k", split="train")

# Apply the formatting function to every example in the dataset.
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset.column_names,
)

# Print the formatted training text of the first example
print(dataset[0]["text"])

README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

Map:   0%|          | 0/49626 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Help me set up my daily to-do list!

Setting up your daily to-do list...<|eot_id|><|start_header_id|>assistant<|end_header_id|>

```python
tasks = []
while True:
    task = input('Enter a task or type 'done' to finish: ')
    if task == 'done': break
    tasks.append(task)
print(f'Your to-do list for today: {tasks}')
```<|eot_id|>


# **Finetune the Model** 

In [23]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
import json
import os

max_seq_length = 2048

# Callback that automatically uploads LoRA checkpoint files to your HuggingFace repo during training.
callback = AutoUploadCallback(
    repo_id="fredrikschultz/flytech_lora"
)

# Create a supervised fine-tuning trainer for LoRA training.
trainer = SFTTrainer(
    model = model,                     # The base model with LoRA layers attached
    tokenizer = tokenizer,             # Tokenizer with chat template applied
    train_dataset = dataset,           # The formatted training dataset
    dataset_text_field = "text",       # Field containing the training text
    max_seq_length = max_seq_length,   # Maximum sequence length
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),  # Handles padding + batching
    dataset_num_proc = 2,              # Multiprocessing for faster dataset loading
    packing = False,                   # Do not pack multiple samples together
    args = TrainingArguments(
        per_device_train_batch_size = 2,       # Number of examples per GPU batch
        gradient_accumulation_steps = 2,       # Accumulate gradients 
        warmup_steps = 5,                      # Warmup for the learning rate scheduler
        num_train_epochs = 1,                  # Number of training epochs
        learning_rate = 2e-4,                  # LoRA learning rate
        fp16 = not is_bfloat16_supported(),    # Use FP16 if BF16 not available
        bf16 = is_bfloat16_supported(),        # Use BF16 when supported by GPU (faster + stable)
        logging_steps = 9999999,               # Disable wandb logging noise
        seed = 3407,                           # Training seed for reproducibility
        save_strategy = "steps",               # Save model every N steps
        save_steps = 150,                      # Save checkpoint after every 150 steps
    ),
    report_to = "none",               # Disable WandB logging
    output_dir = "lora_output",       # Where to save LoRA checkpoints locally
    callbacks=[callback],             # Automatically upload checkpoints to HF
)

# Ensure the model only computes loss on assistant responses,
# not on the user messages. This improves alignment and reduces noise.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/49626 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/49626 [00:00<?, ? examples/s]

In [24]:
# we did not want to lof to wandb so we needed this set up to not run into erros 

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_PROJECT"] = "disabled"
os.environ["WANDB_INIT_TIMEOUT"] = "0"

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49,626 | Num Epochs = 1 | Total steps = 6,204
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).



🤖 Uploading trainer_output/checkpoint-150 to HuggingFace repo root...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).



🤖 Uploading trainer_output/checkpoint-4950 to HuggingFace repo root...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Successfully uploaded to repo root!



In [2]:
rm -rf ~/.cache/huggingface/hub

In [3]:
import os
from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel

# Download the LoRA-fine-tuned model files from your Hugging Face repo
base_path = snapshot_download("fredrikschultz/flytech_lora")

# Load the base model and tokenizer using Unsloth's optimized FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    base_path,
    max_seq_length=2048,
    load_in_4bit=False,
)

# Convert the model to GGUF format and upload it to Hugging Face Hub
model.push_to_hub_gguf(
    "fredrikschultz/lora_python_converter",
    tokenizer,
    quantization_method="q2_k",
    token=huggingface_token,
)

ModuleNotFoundError: No module named 'unsloth'